# 02 — Exploration

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import altair as alt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
card           = pd.read_csv(PROCESSED / "card.csv",           parse_dates=["card_produced_time"])
transaction    = pd.read_csv(PROCESSED / "transaction.csv",    parse_dates=["transaction_time"])
cost_structure = pd.read_csv(PROCESSED / "cost_structure.csv")
rates          = pd.read_csv(PROCESSED / "rates.csv")
print(f"Loaded:  card={len(card):,}  transaction={len(transaction):,}  cost_structure={len(cost_structure):,}  rates={len(rates):,}")

## 1. Transaction observation window

In [ ]:
tx_min = transaction["transaction_time"].min()
tx_max = transaction["transaction_time"].max()
span_days = (tx_max - tx_min).days
print(f"tx_min:    {tx_min.date()}")
print(f"tx_max:    {tx_max.date()}")
print(f"span:      {span_days} days  (≈ {span_days/30:.1f} months)")

## 2. Card production date distribution

In [ ]:
card["cohort_month"] = card["card_produced_time"].dt.to_period("M")
cohort_dist = (card.groupby("cohort_month").size()
                   .rename("cards_produced")
                   .reset_index())
cohort_dist["cohort_month_str"] = cohort_dist["cohort_month"].astype(str)
cohort_dist = cohort_dist[["cohort_month_str", "cards_produced"]]  # drop Period (Altair can't serialize)
cohort_dist

In [ ]:
chart = (alt.Chart(cohort_dist).mark_bar().encode(
    x=alt.X("cohort_month_str:O", title="Card production month"),
    y=alt.Y("cards_produced:Q", title="Cards produced"),
    tooltip=["cohort_month_str", "cards_produced"],
).properties(width=420, height=240, title="Card production by month"))
chart

## 3. Card eligibility per tenure month

In [ ]:
tx_min_m = pd.Period("2018-01", freq="M")
tx_max_m = pd.Period("2018-04", freq="M")

at_risk = []
for t in range(7):
    mask = (
        (card["cohort_month"] + t >= tx_min_m) &
        (card["cohort_month"] + t <= tx_max_m)
    )
    at_risk.append({
        "month_since_issuance":  t,
        "cohort_at_risk":        int(mask.sum()),
    })
at_risk_df = pd.DataFrame(at_risk)
at_risk_df

In [ ]:
# Visualise the cards-at-risk curve
e_chart = (alt.Chart(at_risk_df).mark_line(point=True).encode(
    x=alt.X("month_since_issuance:O", title="Month since issuance (T)"),
    y=alt.Y("cohort_at_risk:Q", title="Cards at risk"),
    tooltip=["month_since_issuance", "cohort_at_risk"],
).properties(width=420, height=240, title="Cards at risk per tenure month (calendar)"))
e_chart

## 5. Cards per market distribution

In [ ]:
market_cards = (card.groupby("profile_address_country").size()
                    .rename("n_cards").sort_values(ascending=False).reset_index())
market_cards["cumulative_pct"] = (market_cards["n_cards"].cumsum() / market_cards["n_cards"].sum() * 100).round(1)
print(f"Total markets: {len(market_cards)}")
market_cards.head(20)

In [ ]:
# Visualise: cards per market, with thresholds annotated
top20 = market_cards.head(20)
m_chart = (alt.Chart(top20).mark_bar().encode(
    x=alt.X("profile_address_country:N",
            sort=alt.SortField("n_cards", order="descending"),
            title="Market"),
    y=alt.Y("n_cards:Q", title="Cards"),
    tooltip=["profile_address_country", "n_cards", "cumulative_pct"],
).properties(width=520, height=240, title="Cards per market (top 20)"))
m_chart

In [ ]:
# How many markets pass each threshold?
for threshold in (10, 30, 60, 100, 200, 500):
    n = (market_cards["n_cards"] >= threshold).sum()
    pct_book = market_cards.loc[market_cards["n_cards"] >= threshold, "n_cards"].sum() / market_cards["n_cards"].sum() * 100
    print(f"  ≥{threshold:>4d} cards: {n:>3d} markets, covering {pct_book:>5.1f}% of the book")

## 6. Currency exposure

In [ ]:
# Cross-currency share among SUCCESS rows
success = transaction[transaction["state"] == "SUCCESS"]
cross_ccy = (success["amount_currency"] != success["billing_amount_currency"])
print(f"SUCCESS transactions:        {len(success):,}")
print(f"  same-currency:             {(~cross_ccy).sum():,}  ({100*(~cross_ccy).mean():.1f}%)")
print(f"  cross-currency:            {cross_ccy.sum():,}  ({100*cross_ccy.mean():.1f}%)")

In [ ]:
# Top currencies on each side
top_amt = success["amount_currency"].value_counts().head(8).rename("amount_count")
top_bil = success["billing_amount_currency"].value_counts().head(8).rename("billing_count")
pd.concat([top_amt, top_bil], axis=1).fillna(0).astype(int)

## 7. Transaction-type × cost-region mix

In [ ]:
crosstab = pd.crosstab(
    success["transaction_type"],
    success["cost_region"],
    margins=True, margins_name="Total",
)
crosstab

In [ ]:
# As percentages of total
pct_crosstab = (crosstab / crosstab.loc["Total","Total"] * 100).round(1)
pct_crosstab